<a href="https://colab.research.google.com/github/nuuuuurinn/Assessing-Racial-Disparities-in-Maternity-Care-with-Machine-Learning/blob/main/DS340W.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Scratch code used for project, used ChatGPT for assistance.

In [ ]:
# to predict the likelihood (probability) of a pregnancy-related death
# to replicate "2.8x" and "3.8x" findings
# by using ML models to calculate Adjusted Odds Ratios

# rates of other racial and ethnic groups could not be included, limiting the interpretation of the study results
# so we use 5.0 to represent suppressed data

In [11]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf

# Load files
df_natality = pd.read_excel("Natality 2018-2024 expanded.xlsx")
df_mortality = pd.read_excel("Multiple Cause of Death, 2018-2024, Single Race.xlsx")

nat = df_natality[['State of Residence', "Mother's Single Race 6", 'Age of Mother 9', 'Births']].copy()
nat.columns = ['state', 'race', 'age', 'births']

mort = df_mortality[['State', 'Single Race 6', 'Five-Year Age Groups', 'Deaths']].copy()
mort.columns = ['state', 'race', 'age', 'deaths']

dataset = pd.merge(nat, mort, on=['state', 'race', 'age'], how ='left')
dataset['deaths'] = dataset['deaths'].fillna(5)

formula = "deaths ~ C(race, Treatment('White')) + C(age)"

model = smf.glm(
    formula = formula,
    data = dataset,
    family=sm.families.Poisson(),
    offset=np.log(dataset['births'])
).fit()

params = model.params
conf = model.conf_int()
conf['aRR'] = params
conf.columns = ['CI_Low', 'CI_High', 'aRR']

results = np.exp(conf).filter(like = 'race', axis = 0)

results = results.sort_values(by = 'aRR', ascending = False)
results.reset_index(inplace = True)
results.rename(columns={'index': 'Race (vs. White)'}, inplace=True)

print("=== Adjusted Risk Ratios (Controlling for Age) ===")
print(results.to_markdown(index=False))

# 1. Sum up all births and deaths by race (ignoring state for the national total)
national_stats = dataset.groupby('race').agg({'births': 'sum', 'deaths': 'sum'}).reset_index()

# 2. Calculate the Rate per 100,000 (The 'ASR' equivalent)
national_stats['rate_per_100k'] = (national_stats['deaths'] / national_stats['births']) * 100000

# 3. Calculate the Disparity Ratio (compared to White)
white_rate = national_stats.loc[national_stats['race'] == 'White', 'rate_per_100k'].values[0]
national_stats['disparity_ratio'] = national_stats['rate_per_100k'] / white_rate

# 4. Sort and show the results
print(national_stats[['race', 'rate_per_100k', 'disparity_ratio']].sort_values('rate_per_100k', ascending=False))

=== Adjusted Risk Ratios (Controlling for Age) ===
| Race (vs. White)                                                         |   CI_Low |   CI_High |      aRR |
|:-------------------------------------------------------------------------|---------:|----------:|---------:|
| C(race, Treatment('White'))[T.Native Hawaiian or Other Pacific Islander] | 49.4717  |  55.6277  | 52.4595  |
| C(race, Treatment('White'))[T.American Indian or Alaska Native]          | 20.5913  |  23.0253  | 21.7743  |
| C(race, Treatment('White'))[T.More than one race]                        |  7.63982 |   8.49656 |  8.05681 |
| C(race, Treatment('White'))[T.Asian]                                     |  2.92303 |   3.2462  |  3.08038 |
| C(race, Treatment('White'))[T.Black or African American]                 |  2.77576 |   3.01163 |  2.89129 |
                                        race  rate_per_100k  disparity_ratio
4  Native Hawaiian or Other Pacific Islander    1482.197690        48.144595
0           Americ

In [10]:
# 1. Prepare the deaths column (fill suppressed NaNs with 0 to avoid the "21x" error)
dataset['deaths_clean'] = dataset['deaths'].fillna(0)

# 2. Create the "Standard Weights"
# This calculates what percentage of ALL births come from each age group in your data
weights = dataset.groupby('age')['births'].sum() / dataset['births'].sum()

# 3. Calculate the Crude Rate for every specific group (Race + Age)
# We group by race and age, sum the deaths and births, then get the rate
grouped = dataset.groupby(['race', 'age']).agg({'deaths_clean': 'sum', 'births': 'sum'}).reset_index()
grouped['rate_per_100k'] = (grouped['deaths_clean'] / grouped['births']) * 100000

# 4. Apply the Weights to get the Age-Standardized Rate (ASR)
# This "adjusts" the rate so that every race is compared as if they had the same age mix
asr_data = pd.merge(grouped, weights.rename('weight'), on='age')
asr_data['weighted_rate'] = asr_data['rate_per_100k'] * asr_data['weight']

# 5. Final Results Table
final_asr = asr_data.groupby('race')['weighted_rate'].sum().reset_index()
final_asr.columns = ['Race', 'ASR (per 100k)']

# Sort by the highest mortality rate
print("=== Replicated Age-Standardized Rates (ASR) ===")
print(final_asr.sort_values('ASR (per 100k)', ascending=False).to_markdown(index=False))

=== Replicated Age-Standardized Rates (ASR) ===
| Race                                      |   ASR (per 100k) |
|:------------------------------------------|-----------------:|
| Native Hawaiian or Other Pacific Islander |        1517.37   |
| American Indian or Alaska Native          |         712.418  |
| More than one race                        |         255.484  |
| Asian                                     |         217.875  |
| Black or African American                 |          95.7386 |
| White                                     |          31.6523 |


In [ ]:
import pandas as pd

# 1. Data from the Maryland SMM Paper (2016-2019)
# Metric: Unadjusted Relative Risk (RR) for SMM compared to Non-Hispanic White women
smm_data = {
    'Race_Ethnicity': ['Non-Hispanic Black', 'Non-Hispanic Other', 'Hispanic', 'Non-Hispanic White'],
    'SMM_Relative_Risk': [2.03, 1.27, 1.18, 1.00] # Baseline is 1.00
}
df_smm = pd.DataFrame(smm_data)

# 2. Data from the US Pregnancy-Related Deaths Paper (2018-2022)
# Metric: Age-Standardized Rate (ASR) per 100,000 live births
mortality_data = {
    'Race_Ethnicity': ['American Indian / Alaska Native', 'Non-Hispanic Black', 'Non-Hispanic White', 'Hispanic', 'Non-Hispanic Asian'],
    'Mortality_ASR_per_100k': [106.3, 76.9, 27.6, 25.9, 21.8]
}
df_mortality = pd.DataFrame(mortality_data)

# Calculate Mortality Relative Risk (Baseline = Non-Hispanic White ASR of 27.6)
baseline_mortality = 27.6
df_mortality['Mortality_Relative_Risk'] = round(df_mortality['Mortality_ASR_per_100k'] / baseline_mortality, 2)

# 3. Combine the datasets based on Race/Ethnicity
# Using an outer join to capture groups that might only appear in one dataset
df_combined = pd.merge(df_smm, df_mortality, on='Race_Ethnicity', how='outer')

# 4. Create a "Composite Disparity Score" (Average of available Relative Risks)
# This gives us a single number to rank the overall severity of the disparity.
df_combined['Composite_Disparity_Score'] = df_combined[['SMM_Relative_Risk', 'Mortality_Relative_Risk']].mean(axis=1, skipna=True)

# 5. Rank the results from most severe disparity to least
df_ranked = df_combined.sort_values(by='Composite_Disparity_Score', ascending=False).reset_index(drop=True)

# Display the final ranked table
print("=== Ranked Racial Disparities in Maternal Healthcare (Combined) ===")
print(df_ranked[['Race_Ethnicity', 'SMM_Relative_Risk', 'Mortality_Relative_Risk', 'Composite_Disparity_Score']])

=== Ranked Racial Disparities in Maternal Healthcare (Combined) ===
                    Race_Ethnicity  SMM_Relative_Risk  \
0  American Indian / Alaska Native                NaN   
1               Non-Hispanic Black               2.03   
2               Non-Hispanic Other               1.27   
3                         Hispanic               1.18   
4               Non-Hispanic White               1.00   
5               Non-Hispanic Asian                NaN   

   Mortality_Relative_Risk  Composite_Disparity_Score  
0                     3.85                       3.85  
1                     2.79                       2.41  
2                      NaN                       1.27  
3                     0.94                       1.06  
4                     1.00                       1.00  
5                     0.79                       0.79  
